<a href="https://colab.research.google.com/github/acastellanos-ie/NLP-MBDS-EN-NEW/blob/main/05_language_modeling/language_modeling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Language Modeling and Decoding

This notebook opens the black box of a decoder-only language model. A model like this does not think or plan a paragraph; its only real operation is to compute a probability distribution over the next token. We build the auto-regressive loop by hand, steer it with sampling and temperature, and measure it with perplexity.

This is the engine room of the Helcon Support Copilot. Every later piece, the triage classifier, the retriever, and the agent, sits on top of the next-token machine we examine here. We also use perplexity as an anomaly signal over Helcon's real tickets, our first contact with the shared SDK (system_io).

In [ ]:
# Setup, run this cell first.
# It installs the libraries and pre-downloads the two models of the session (~1.3 GB),
# so every later cell loads from cache instantly.
!pip install -q transformers torch matplotlib numpy

from huggingface_hub import snapshot_download
for repo in ["gpt2", "HuggingFaceTB/SmolLM2-360M-Instruct"]:
    snapshot_download(repo)
    print(f"cached: {repo}")
# Repo access so the shared Helcon SDK (system_io.load_seed) is importable.
REPO = "https://github.com/acastellanos-ie/NLP-MBDS-EN-NEW"
import os, sys
if not os.path.exists("NLP-MBDS-EN") and not os.path.exists("../system/system_io.py"):
    !git clone --depth 1 -q $REPO NLP-MBDS-EN
for cand in ("../system", "NLP-MBDS-EN/system", "system"):
    if os.path.exists(os.path.join(cand, "system_io.py")):
        sys.path.insert(0, cand); break
from system_io import load_seed

print("Setup complete.")

Locally the repo is already on disk; on Colab we clone it so the shared Helcon SDK (`load_seed`) imports either way.

## Step 1: Loading the model

We use GPT-2 (2019, 124M parameters). It is small and old on purpose. It shares the same paradigm as GPT-4 or Llama, a Transformer decoder trained on next-token prediction, but it is weak enough that we can watch the machinery without being dazzled by the output. At the end of the session we meet its 2024 great-grandchild.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import matplotlib.pyplot as plt
import numpy as np

model_id = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_id)   # instant; the setup cell already cached it
model = AutoModelForCausalLM.from_pretrained(model_id)

# Set to evaluation mode (turns off dropout layers for deterministic outputs)
model.eval()
print(f"{model_id} loaded. Vocabulary: {tokenizer.vocab_size:,} tokens (BPE pieces, not words)")

## Step 2: The next-token distribution

Let's feed the model an unfinished sentence. We don't want it to generate text yet. We want to see the raw scores (logits) it assigns to every token in its vocabulary for the next position.

Before running: the prompt is *"The name of the capital of France is"*. How much probability would you give to `" Paris"` as the very next token? Most people guess far too high.

In [ ]:
PROMPT = "The name of the capital of France is"   # also try: the same prompt in lowercase,
TOP_K  = 5                                        # or "I walked into the bank and" (two senses fight)

input_ids = tokenizer.encode(PROMPT, return_tensors="pt")
with torch.no_grad():
    outputs = model(input_ids)

# The model returns logits for EVERY position; we only care about the last one
next_token_logits = outputs.logits[0, -1, :]
probabilities = torch.nn.functional.softmax(next_token_logits, dim=-1)
top_probs, top_indices = torch.topk(probabilities, TOP_K)
top_tokens = [tokenizer.decode([idx]) for idx in top_indices]

plt.figure(figsize=(8, 4))
plt.barh(top_tokens[::-1], top_probs.numpy()[::-1] * 100, color="skyblue")
plt.xlabel("Probability (%)")
plt.title(f"Next-token distribution for: '{PROMPT}'")
for i, v in enumerate(top_probs.numpy()[::-1] * 100):
    plt.text(v + 0.5, i, f"{v:.2f}%", va="center")
plt.show()

## Takeaway

`" Paris"` tops the distribution on every run, but with much less probability than most people expect. A good chunk of the mass goes to grammatical continuations like `" the"` or `" not"`. The exact percentage drifts with the library version; what matters is the gap to the second candidate. The model "knows" the answer not because it has a database of capitals, but because in its training data *Paris* statistically followed these exact words. That statistical reflex is the entire machine.

## Step 3: Auto-regressive generation from scratch

Now we build the engine ourselves:

1. Predict the next token.
2. Take the most probable one (**greedy decoding**) and append it to the prompt.
3. Feed the new, longer prompt back into the model. Repeat.

The repetition penalty dampens tokens that already appeared. Before touching anything else, try `REPETITION_PENALTY = 1.0`, that is, no penalty at all, and watch what pure greedy decoding does. Then try something absurd like 5.0.

In [ ]:
PROMPT             = "The name of the capital of France is"
MAX_NEW_TOKENS     = 15      # each step re-feeds the whole sequence so far
REPETITION_PENALTY = 1.2     # try: 1.0 (off), 1.2, 5.0

current_input_ids = tokenizer.encode(PROMPT, return_tensors="pt")
print(f"Starting prompt: '{PROMPT}'")
print("-" * 40)
for step in range(MAX_NEW_TOKENS):
    with torch.no_grad():
        step_logits = model(current_input_ids).logits[0, -1, :]

    # dampen every token already in the sequence (once per unique token)
    for token_id in set(current_input_ids[0].tolist()):
        if step_logits[token_id] > 0:
            step_logits[token_id] /= REPETITION_PENALTY
        else:
            step_logits[token_id] *= REPETITION_PENALTY

    next_token_id = torch.argmax(step_logits)
    print(f"Step {step + 1:>2}: '{tokenizer.decode([next_token_id])}'")
    current_input_ids = torch.cat([current_input_ids, next_token_id.view(1, 1)], dim=-1)

print("-" * 40)
print("Final output:\n" + tokenizer.decode(current_input_ids[0]))

## Takeaway

The loop is the whole secret: predict, append, repeat. There is no plan and no sentence-level intent. With the penalty off, greedy decoding locks into a repeating loop: it is a deterministic map from context to next token, and once the context becomes periodic the output is trapped forever. At 5.0 the opposite failure appears: the model is effectively banned from reusing function words like *the* or *of*, and grammar disintegrates. Decoding settings are not cosmetic. They are a product decision about which failure mode you prefer.

## Step 4: Sampling and temperature

Everything so far is deterministic: same prompt, same output, every time. One change breaks that spell. Instead of always taking the argmax, we **sample** from the distribution, softened by a temperature:

```python
probs = softmax(step_logits / TEMPERATURE)
next_token_id = torch.multinomial(probs, num_samples=1)
```

That is the entire difference between "boring autocomplete" and "creative AI". The cell below runs the same loop twice, same code, same prompt.

In [ ]:
def generate_with_sampling(prompt, max_new_tokens=15, temperature=0.8, penalty=1.2):
    ids = tokenizer.encode(prompt, return_tensors="pt")
    for _ in range(max_new_tokens):
        with torch.no_grad():
            logits = model(ids).logits[0, -1, :]
        for token_id in set(ids[0].tolist()):
            if logits[token_id] > 0:
                logits[token_id] /= penalty
            else:
                logits[token_id] *= penalty
        probs = torch.nn.functional.softmax(logits / temperature, dim=-1)
        next_id = torch.multinomial(probs, num_samples=1)
        ids = torch.cat([ids, next_id.view(1, 1)], dim=-1)
    return tokenizer.decode(ids[0])

# try: change the two temperatures - low is cautious, high is adventurous
print("--- low temperature (0.3) ---")
print(generate_with_sampling("The name of the capital of France is", temperature=0.3))
print()
print("--- high temperature (1.5) ---")
print(generate_with_sampling("The name of the capital of France is", temperature=1.5))

Two different texts from identical code: that's sampling. But what is the temperature *doing*, exactly? It is not an adjective, it is arithmetic: divide the logits by *T* before the softmax. Watch what that does to the **same** distribution:

In [ ]:
TEMPS = [0.1, 0.7, 1.0, 1.5, 3.0]

ids = tokenizer.encode("The name of the capital of France is", return_tensors="pt")
with torch.no_grad():
    logits = model(ids).logits[0, -1, :]

top_ids = torch.topk(logits, 12).indices
labels = [tokenizer.decode([i]) for i in top_ids]

fig, axes = plt.subplots(1, len(TEMPS), figsize=(16, 3.4), sharey=True)
for ax, T in zip(axes, TEMPS):
    p = torch.nn.functional.softmax(logits / T, dim=-1)[top_ids]
    ax.bar(range(len(top_ids)), p.numpy() * 100, color="salmon")
    ax.set_title(f"T = {T}")
    ax.set_xticks(range(len(top_ids)))
    ax.set_xticklabels(labels, rotation=75, fontsize=7)
axes[0].set_ylabel("probability (%)")
plt.suptitle("The SAME logits, only the temperature changes")
plt.tight_layout()
plt.show()

At T = 0.1 the distribution collapses onto the top token (almost greedy); at T = 3 it is nearly flat, and thousands of unrelated tokens become plausible. Grammar lives in the *sharpness* of the distribution. Flatten it too much and language itself dissolves. "More temperature = more creativity" is marketing; past a threshold it is just noise.

The cell below is the full control panel, the same knobs every commercial LLM API exposes. **Top-k** and **top-p** don't reshape the distribution; they *truncate its tail* before sampling, which is why they can tame a high temperature. Try temperature 3.0 with and without `top_p = 0.9` and see the difference.

In [ ]:
PROMPT         = "The name of the capital of France is"
STRATEGY       = "temperature"   # greedy | temperature | top_k | top_p | combined
TEMPERATURE    = 0.8             # used by: temperature, combined
TOP_K          = 50              # used by: top_k, combined; sample only among the K best tokens
TOP_P          = 0.9             # used by: top_p, combined; smallest token set covering P of the mass
MAX_NEW_TOKENS = 25
SEED           = 42              # same seed + same knobs = same text on every machine

gen_args = dict(max_new_tokens=MAX_NEW_TOKENS, repetition_penalty=1.1,
                pad_token_id=tokenizer.eos_token_id)
if STRATEGY == "greedy":
    gen_args.update(do_sample=False)
elif STRATEGY == "temperature":
    gen_args.update(do_sample=True, temperature=TEMPERATURE)
elif STRATEGY == "top_k":
    gen_args.update(do_sample=True, top_k=TOP_K)
elif STRATEGY == "top_p":
    gen_args.update(do_sample=True, top_p=TOP_P, top_k=0)
elif STRATEGY == "combined":
    gen_args.update(do_sample=True, temperature=TEMPERATURE, top_k=TOP_K, top_p=TOP_P)

torch.manual_seed(SEED)
inputs = tokenizer(PROMPT, return_tensors="pt")
out = model.generate(**inputs, **gen_args)
print(f"[strategy: {STRATEGY}]")
print(tokenizer.decode(out[0], skip_special_tokens=True))

## Takeaway

Greedy is fully deterministic; sampling trades determinism for variety; top-k and top-p cut the tail of the distribution before sampling. With the same `SEED` and knobs, every machine prints the same text. One trap worth remembering: `temperature` does **nothing** unless `do_sample=True`, a silent no-op that we will meet again in the RAG and agents sessions. These five knobs are exactly what OpenAI, Anthropic and Gemini expose in their APIs; now you know what each one does to the math.

## Step 5: Measuring the model with perplexity

A whole session generating text and not a single number measuring the model. Time to fix that. **Perplexity** is the standard metric of language modeling: the exponential of the average per-token surprise. Low means the text is what the model expects; high means the model is shocked.

Three test sentences: a Shakespeare quote, a line of corporate boilerplate, and the same quote with its words shuffled. Think for a second about which one GPT-2 will find *least* surprising. The answer says something uncomfortable about corporate language.

In [ ]:
import torch.nn.functional as F
from IPython.display import display, HTML

def perplexity(text):
    ids = tokenizer.encode(text, return_tensors="pt")
    with torch.no_grad():
        out = model(ids, labels=ids)        # loss = average next-token negative log-likelihood
    return torch.exp(out.loss).item()

def surprisal_heatmap(text):
    """Render each token colored by how surprised the model was to see it (white = expected, red = shocking)."""
    ids = tokenizer.encode(text, return_tensors="pt")
    if ids.shape[1] < 2:
        print("(need at least two tokens)")
        return
    with torch.no_grad():
        logits = model(ids).logits
    logprobs = F.log_softmax(logits[0, :-1], dim=-1)
    surprisal = (-logprobs[range(ids.shape[1] - 1), ids[0, 1:]]).tolist()
    hi = max(surprisal) or 1.0
    spans = [f"<span>{tokenizer.decode([ids[0, 0]])}</span>"]
    for tok, s in zip(ids[0, 1:], surprisal):
        spans.append(f"<span style='background-color: rgba(255,60,60,{min(s / hi, 1.0):.2f}); "
                     f"padding:1px'>{tokenizer.decode([tok])}</span>")
    display(HTML("<div style='font-size:16px; line-height:1.9'>" + "".join(spans) + "</div>"))

SENTENCES = [
    "To be, or not to be, that is the question.",
    "We leverage synergies to deliver value-added solutions for our stakeholders.",
    "Question that the is to be or be to not.",
]
for s in sorted(SENTENCES, key=perplexity):
    print(f"perplexity = {perplexity(s):>9.1f}   {s}")

The shuffled sentence explodes: same words, destroyed order, so perplexity is clearly measuring *sequence* probability, not vocabulary. And the corporate boilerplate scores almost as low as Shakespeare: it is statistically ultra-predictable language, which is precisely what makes it sound hollow.

Now the important experiment. Two sentences, one word apart, one true and one false:

In [ ]:
for s in ["The capital of France is Paris.", "The capital of France is Rome."]:
    print(f"perplexity = {perplexity(s):.1f}   {s}")
    surprisal_heatmap(s)

And now on our own data: which of Helcon's real support tickets does GPT-2 find most surprising? Read perplexity as an anomaly signal over the ticket stream. The noisiest tickets are the ones worth a human's eye first.

In [ ]:
# A real Helcon support ticket; perplexity as an anomaly signal over the ticket stream.
SENTENCE = load_seed("tickets")[0]["text"]   # try: another ticket, e.g. load_seed("tickets")[7]["text"]
# try: hardcode one, e.g. "Good afternoon, The app freezes since yesterday whenever I search. Thanks in advance."

print(SENTENCE)
print(f"perplexity = {perplexity(SENTENCE):.1f}")
surprisal_heatmap(SENTENCE)


In [ ]:
# Perplexity over 5 sampled tickets, most surprising first.
import random
random.seed(0)   # try: change the seed to sample a different five
sample = random.sample(load_seed("tickets"), 5)
for t in sorted(sample, key=lambda r: perplexity(r["text"]), reverse=True):
    print(f"{perplexity(t['text']):>9.1f}  {t['message_id']}  {t['text']}")


## Takeaway

The fluent lie scores a sentence-level perplexity close to the truth's. Only the heatmap shows *Rome* glowing red, a single surprising token drowned by a fluent average. This is the most business-relevant fact of the session: **perplexity measures fluency, not truth. Hallucination is not a bug in the math, it is the math.** A model can be maximally confident and completely wrong, as long as the wrongness is well written. (Methodological footnote: perplexities are only comparable within the same model and tokenizer, never across models.)

## Step 6: 2019 vs 2024, the same loop five years later

Everything we built today, the distribution, the loop, the sampling, and the perplexity, is exactly what runs inside every modern LLM. So what changed in five years? Scale, data, and **instruction tuning**. The same request now hits GPT-2 (raw next-token reflex) and SmolLM2-360M-Instruct, a 2024 pocket model only about 3x GPT-2's size:

In [ ]:
MODEL_ID    = "HuggingFaceTB/SmolLM2-360M-Instruct"   # fallback if weak: "HuggingFaceTB/SmolLM2-1.7B-Instruct"
INSTRUCTION = "List three European capitals."

small_tok = AutoTokenizer.from_pretrained(MODEL_ID)     # instant; cached by the setup cell
small_model = AutoModelForCausalLM.from_pretrained(MODEL_ID)
small_model.eval()

chat = small_tok.apply_chat_template([{"role": "user", "content": INSTRUCTION}],
                                     tokenize=False, add_generation_prompt=True)
ids = small_tok(chat, return_tensors="pt")
with torch.no_grad():
    out = small_model.generate(**ids, max_new_tokens=80, do_sample=False,
                               pad_token_id=small_tok.eos_token_id)
print("=== SmolLM2-360M-Instruct (2024) ===")
print(small_tok.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True))

g_in = tokenizer(INSTRUCTION, return_tensors="pt")
g_out = model.generate(**g_in, max_new_tokens=40, do_sample=False, repetition_penalty=1.1,
                       pad_token_id=tokenizer.eos_token_id)
print("\n=== GPT-2 (2019) ===")
print(tokenizer.decode(g_out[0], skip_special_tokens=True))

## Takeaway

The instruct model *answers you*; GPT-2 *continues your text*. It treats the instruction as a sentence to autocomplete, because that is all a raw language model does. Both run the identical loop we built by hand in step 3. The difference is scale, data, and a second training phase that teaches the model to treat input as a request: instruction tuning, which belongs to the transfer learning story of the next session. One repeated operation, sample the next token, plus five years of engineering: that is "Generative AI".